In [10]:
import re
import numpy as np
from collections import Counter
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Original dataset from lab
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

# Text preprocessing
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # remove punctuation/numbers
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(words)

cleaned_dataset = [preprocess(doc) for doc in dataset]

# =========================
# TF-IDF with preprocessing
# =========================
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(cleaned_dataset)

k = 2
km_tfidf = KMeans(n_clusters=k, random_state=42)
km_tfidf.fit(X_tfidf)

y_pred_tfidf = km_tfidf.predict(X_tfidf)

# Purity calculation
cluster_counts_tfidf = Counter(y_pred_tfidf)
purity_tfidf = max(cluster_counts_tfidf.values()) / len(y_pred_tfidf)

print("TF-IDF Predicted Clusters:", y_pred_tfidf)
print("TF-IDF Purity:", purity_tfidf)

# ============================
# Word2Vec with preprocessing
# ============================
tokenized_dataset = [doc.split() for doc in cleaned_dataset]

word2vec_model = Word2Vec(
    sentences=tokenized_dataset,
    vector_size=100,
    window=5,
    min_count=1,
    workers=1
)

X_w2v = np.array([
    np.mean([word2vec_model.wv[word] for word in doc if word in word2vec_model.wv], axis=0)
    for doc in tokenized_dataset
])

km_w2v = KMeans(n_clusters=k, random_state=42)
km_w2v.fit(X_w2v)

y_pred_w2v = km_w2v.predict(X_w2v)

cluster_counts_w2v = Counter(y_pred_w2v)
purity_w2v = max(cluster_counts_w2v.values()) / len(y_pred_w2v)

print("Word2Vec Predicted Clusters:", y_pred_w2v)
print("Word2Vec Purity:", purity_w2v)

TF-IDF Predicted Clusters: [1 0 1 1 1]
TF-IDF Purity: 0.8
Word2Vec Predicted Clusters: [0 0 1 0 1]
Word2Vec Purity: 0.6
